<a href="https://colab.research.google.com/github/chloe8407/gpt-2.0-sherlock/blob/main/5_Single_Head_Attention_Sherlock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5. Single-Head Masked Self-Attention — Sherlock Holmes

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

URL = "https://www.gutenberg.org/cache/epub/1661/pg1661.txt"
if not Path("sherlock.txt").exists():
    !wget -q -O sherlock.txt {URL}

raw = open("sherlock.txt", "r", encoding="utf-8").read()
# Project Gutenberg 라이선스 헤더/푸터 제거 -> 본문만 학습
start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"
s = raw.find(start_marker)
e = raw.find(end_marker)
start = raw.find("\n", s) + 1 if s != -1 else 0
end = e if e != -1 else len(raw)
text = raw[start:end].strip()

chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

class NextTokenDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size
    def __len__(self):
        return len(self.data) - self.block_size
    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

block_size = 32
dataset = NextTokenDataset(data, block_size)
loader = DataLoader(dataset, batch_size=64, shuffle=True)
xb, yb = next(iter(loader))

In [2]:
class SingleHeadSelfAttention(nn.Module):
    def __init__(self, emb_dim, block_size):
        super().__init__()
        self.key = nn.Linear(emb_dim, emb_dim, bias=False)
        self.query = nn.Linear(emb_dim, emb_dim, bias=False)
        self.value = nn.Linear(emb_dim, emb_dim, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        wei = q @ k.transpose(-2, -1) * (C ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)

        out = wei @ v
        return out

In [3]:
class TinyAttentionLM(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=64):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(block_size, emb_dim)
        self.attn = SingleHeadSelfAttention(emb_dim, block_size)
        self.lm_head = nn.Linear(emb_dim, vocab_size)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)
        tok = self.token_embedding(x)
        pos = self.position_embedding(pos)[None]
        h = tok + pos
        h = self.attn(h)
        logits = self.lm_head(h)
        return logits

model = TinyAttentionLM(vocab_size, block_size)
logits = model(xb)
print("logits.shape:", logits.shape)

logits.shape: torch.Size([64, 32, 88])


In [4]:
def sequence_cross_entropy(logits, targets):
    return F.cross_entropy(logits.transpose(1, 2), targets)

def train_one_epoch(model, loader, optimizer, device, max_steps=None):
    model.train()
    total_loss, total_count = 0.0, 0
    for step, (xb, yb) in enumerate(loader):
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = sequence_cross_entropy(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)
        if max_steps is not None and step + 1 >= max_steps:
            break
    return total_loss / total_count

device = "cuda" if torch.cuda.is_available() else "cpu"
model = TinyAttentionLM(vocab_size, block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for epoch in range(100):
    train_loss = train_one_epoch(model, loader, optimizer, device, max_steps=300)
    if epoch % 10 == 0 or epoch == 99:
        print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

epoch  0 | train loss 2.9152
epoch 10 | train loss 2.2757
epoch 20 | train loss 2.2613
epoch 30 | train loss 2.2550
epoch 40 | train loss 2.2497
epoch 50 | train loss 2.2375
epoch 60 | train loss 2.2318
epoch 70 | train loss 2.2332
epoch 80 | train loss 2.2328
epoch 90 | train loss 2.2314
epoch 99 | train loss 2.2319


In [5]:
@torch.no_grad()
def sample_attention_model(model, block_size, stoi, itos, device, start_text="Sherlock Holmes", max_new_tokens=400):
    model.eval()
    context = torch.zeros((1, block_size), dtype=torch.long, device=device)
    for ch in start_text:
        if ch in stoi:
            ix = torch.tensor([[stoi[ch]]], device=device)
            context = torch.cat([context[:, 1:], ix], dim=1)
    out = list(start_text)
    for _ in range(max_new_tokens):
        logits = model(context)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        ix = torch.multinomial(probs, num_samples=1)
        out.append(itos[ix.item()])
        context = torch.cat([context[:, 1:], ix], dim=1)
    return "".join(out)

print(sample_attention_model(model, block_size, stoi, itos, device, start_text="Sherlock Holmes", max_new_tokens=400))

Sherlock Holmesn.”

“The hicetlle
what. Mr. Sttilede my be, widines se hing
at ige bers arvesiun to cswe meed rile ndeding theantomatthe at threcthe, aypach?”

“Hy hin hos the quis he wis Mjrewrocheante obendesknon anick here aidavil sen me
thew argis sapono thandlacer yo hivee dst er chi fre, breed on yo Cro gliters nat sem. Yourdig igr?” saven ofrend poud Whalled dy
ockeras tounden
che wes hamolmanl
yo houd I 
